# Creator Bandit Playbook: Content Publishing Strategy

**Goal:** Implement the complete Creator Bandit Playbook from the Medium article.

**What you'll learn:** How to use bandits for content strategy with exploration budgets and arm retirement.

**Scenario:** You're a content creator testing 6 content formats (3 topics × 2 formats). Which should you publish more often?

**Playbook phases:**
1. **Weeks 1-3:** Pure exploration (try everything)
2. **Weeks 4-26:** Tilt toward winners (80% exploit, 20% explore)
3. **Week 26:** Retire worst performers, add new experiments
4. **Weeks 27-52:** Continue with refined options

---

## Setup

In [ ]:
!pip install -q numpy matplotlib pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)

## Define 6 Content Arms

Each arm = Topic × Format combination with realistic engagement distributions.

In [ ]:
# Define content arms: (name, mean_engagement, std_engagement)
initial_arms = [
    ('AI/Tutorial', 0.065, 0.02),        # Best performer
    ('AI/Opinion', 0.045, 0.015),        # Medium
    ('Finance/Tutorial', 0.055, 0.018),  # Medium-high
    ('Finance/Opinion', 0.035, 0.012),   # Lower
    ('Career/Tutorial', 0.040, 0.014),   # Lower
    ('Career/Opinion', 0.025, 0.010),    # Worst
]

print("Content Strategy Arms:")
print("\n{:<20} {:<15} {:<15}".format('Content Type', 'Avg Engagement', 'Variability'))
print("="*50)
for name, mean, std in initial_arms:
    print(f"{name:<20} {mean:.1%}           ±{std:.1%}")

print("\nNote: These are the TRUE values (unknown to the algorithm)")

## Helper: Simulate Engagement

Function to simulate engagement for a given arm.

In [ ]:
def get_engagement(arm_params):
    """Simulate engagement (clicks/views ratio) for an arm."""
    name, mean, std = arm_params
    # Engagement is bounded [0, 1]
    engagement = np.random.normal(mean, std)
    return max(0, min(1, engagement))

## Phase 1: Pure Exploration (Weeks 1-3)

Try each arm multiple times to gather initial data. No exploitation yet.

In [ ]:
# Phase 1: Pure exploration
exploration_weeks = 3
posts_per_week = 4
total_exploration_posts = exploration_weeks * posts_per_week

# Initialize tracking
current_arms = initial_arms.copy()
n_arms = len(current_arms)

# Thompson Sampling state: Beta(alpha, beta) for each arm
# We'll convert engagement [0,1] to binary success/failure for Beta-Bernoulli
alpha = np.ones(n_arms)
beta = np.ones(n_arms)
pulls = np.zeros(n_arms)
total_engagement = np.zeros(n_arms)

# History
history = []

print(f"Phase 1: Pure Exploration ({exploration_weeks} weeks, {posts_per_week} posts/week)\n")

# Try each arm equally during exploration
for post in range(total_exploration_posts):
    arm_idx = post % n_arms  # Round-robin
    engagement = get_engagement(current_arms[arm_idx])
    
    pulls[arm_idx] += 1
    total_engagement[arm_idx] += engagement
    
    # Update Beta distribution (treating engagement > 0.05 as "success")
    if engagement > 0.05:
        alpha[arm_idx] += 1
    else:
        beta[arm_idx] += 1
    
    history.append({
        'week': (post // posts_per_week) + 1,
        'post': post + 1,
        'arm': arm_idx,
        'arm_name': current_arms[arm_idx][0],
        'engagement': engagement,
        'phase': 'exploration'
    })

print("Exploration complete! Initial learnings:")
for i, (name, _, _) in enumerate(current_arms):
    avg_eng = total_engagement[i] / pulls[i] if pulls[i] > 0 else 0
    print(f"  {name:<20} : {int(pulls[i])} posts, {avg_eng:.1%} avg engagement")

## Phase 2: Tilt Toward Winners (Weeks 4-26)

Use Thompson Sampling with 20% exploration budget. This means:
- 80% of the time: pick the best arm according to Thompson Sampling
- 20% of the time: explore randomly

In [ ]:
# Phase 2: Tilt toward winners
tilt_weeks = 23  # Weeks 4-26
exploration_rate = 0.20  # 20% exploration budget

print(f"\nPhase 2: Tilt Toward Winners ({tilt_weeks} weeks)")
print(f"Strategy: {100*(1-exploration_rate):.0f}% exploit, {100*exploration_rate:.0f}% explore\n")

for week in range(exploration_weeks + 1, exploration_weeks + tilt_weeks + 1):
    for post in range(posts_per_week):
        # Decide: explore or exploit?
        if np.random.random() < exploration_rate:
            # Explore: random arm
            arm_idx = np.random.randint(n_arms)
        else:
            # Exploit: Thompson Sampling
            theta_samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
            arm_idx = np.argmax(theta_samples)
        
        # Publish and observe engagement
        engagement = get_engagement(current_arms[arm_idx])
        
        pulls[arm_idx] += 1
        total_engagement[arm_idx] += engagement
        
        # Update beliefs
        if engagement > 0.05:
            alpha[arm_idx] += 1
        else:
            beta[arm_idx] += 1
        
        history.append({
            'week': week,
            'post': len(history) + 1,
            'arm': arm_idx,
            'arm_name': current_arms[arm_idx][0],
            'engagement': engagement,
            'phase': 'tilt'
        })

print(f"Week 26 results:")
for i, (name, _, _) in enumerate(current_arms):
    avg_eng = total_engagement[i] / pulls[i] if pulls[i] > 0 else 0
    print(f"  {name:<20} : {int(pulls[i]):3d} posts, {avg_eng:.1%} avg engagement")

## Phase 3: Arm Retirement (Week 26)

Drop the 2 worst-performing arms and add 2 new experiments.

In [ ]:
# Identify worst 2 arms
avg_engagements = [total_engagement[i] / pulls[i] if pulls[i] > 0 else 0 for i in range(n_arms)]
worst_arms = np.argsort(avg_engagements)[:2]

print("\n" + "="*60)
print("WEEK 26: ARM RETIREMENT")
print("="*60)

print("\nRetiring worst performers:")
for idx in worst_arms:
    name = current_arms[idx][0]
    avg = avg_engagements[idx]
    print(f"  ❌ {name} ({avg:.1%} engagement)")

# Add 2 new experimental arms
new_arms = [
    ('Tech/Tutorial', 0.058, 0.017),   # New experiment 1
    ('Productivity/Opinion', 0.042, 0.013)  # New experiment 2
]

print("\nAdding new experiments:")
for name, mean, std in new_arms:
    print(f"  ➕ {name} (unknown engagement)")

# Create new arm list (keep best 4, add 2 new)
keep_arms = [i for i in range(n_arms) if i not in worst_arms]
current_arms = [current_arms[i] for i in keep_arms] + new_arms
n_arms = len(current_arms)

# Reset Thompson Sampling for new arms (keep history for old arms)
new_alpha = np.ones(n_arms)
new_beta = np.ones(n_arms)
new_pulls = np.zeros(n_arms)
new_total_engagement = np.zeros(n_arms)

for new_idx, old_idx in enumerate(keep_arms):
    new_alpha[new_idx] = alpha[old_idx]
    new_beta[new_idx] = beta[old_idx]
    new_pulls[new_idx] = pulls[old_idx]
    new_total_engagement[new_idx] = total_engagement[old_idx]

alpha = new_alpha
beta = new_beta
pulls = new_pulls
total_engagement = new_total_engagement

print(f"\nNew arm lineup ({n_arms} total):")
for i, (name, _, _) in enumerate(current_arms):
    if pulls[i] > 0:
        avg = total_engagement[i] / pulls[i]
        print(f"  {name:<20} : {int(pulls[i]):3d} posts, {avg:.1%} avg engagement")
    else:
        print(f"  {name:<20} : NEW (no data yet)")

## Phase 4: Continue with Refined Options (Weeks 27-52)

In [ ]:
# Phase 4: Continue with new arms
final_weeks = 26  # Weeks 27-52

print(f"\nPhase 4: Refined Strategy ({final_weeks} weeks)\n")

for week in range(27, 53):
    for post in range(posts_per_week):
        # Same strategy: 20% explore, 80% exploit
        if np.random.random() < exploration_rate:
            arm_idx = np.random.randint(n_arms)
        else:
            theta_samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
            arm_idx = np.argmax(theta_samples)
        
        engagement = get_engagement(current_arms[arm_idx])
        
        pulls[arm_idx] += 1
        total_engagement[arm_idx] += engagement
        
        if engagement > 0.05:
            alpha[arm_idx] += 1
        else:
            beta[arm_idx] += 1
        
        history.append({
            'week': week,
            'post': len(history) + 1,
            'arm': arm_idx,
            'arm_name': current_arms[arm_idx][0],
            'engagement': engagement,
            'phase': 'refined'
        })

print("Year-end results:")
for i, (name, _, _) in enumerate(current_arms):
    if pulls[i] > 0:
        avg_eng = total_engagement[i] / pulls[i]
        print(f"  {name:<20} : {int(pulls[i]):3d} posts ({100*pulls[i]/len(history):.1f}%), {avg_eng:.1%} engagement")
    else:
        print(f"  {name:<20} : Not used")

## Visualize Learning Over Time

Show how content selection evolved across all 52 weeks.

In [ ]:
# Convert to DataFrame for analysis
df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Cumulative engagement over time
df['cumulative_engagement'] = df['engagement'].cumsum()
ax1.plot(df['post'], df['cumulative_engagement'], linewidth=2)
ax1.axvline(total_exploration_posts, color='red', linestyle='--', alpha=0.5, label='End of exploration')
ax1.axvline(26 * posts_per_week, color='orange', linestyle='--', alpha=0.5, label='Arm retirement')
ax1.set_xlabel('Post Number', fontsize=12)
ax1.set_ylabel('Cumulative Engagement', fontsize=12)
ax1.set_title('Total Engagement Over 52 Weeks (208 posts)', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Arm selection frequency by week
week_arm_counts = df.groupby(['week', 'arm_name']).size().unstack(fill_value=0)
week_arm_pct = week_arm_counts.div(week_arm_counts.sum(axis=1), axis=0) * 100

# Stacked area chart
week_arm_pct.plot(kind='area', stacked=True, ax=ax2, alpha=0.7)
ax2.axvline(3, color='red', linestyle='--', alpha=0.5, linewidth=2)
ax2.axvline(26, color='orange', linestyle='--', alpha=0.5, linewidth=2)
ax2.set_xlabel('Week', fontsize=12)
ax2.set_ylabel('% of Posts', fontsize=12)
ax2.set_title('Content Mix Evolution: From Exploration to Optimization', fontsize=14)
ax2.set_ylim([0, 100])
ax2.legend(title='Content Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNotice: The algorithm learned to focus on high-engagement content!")

## Compare to Random Publishing Strategy

In [ ]:
# Simulate random strategy (equal probability for all arms)
np.random.seed(42)
random_engagement = []

for _ in range(len(history)):
    # Randomly pick from initial 6 arms (simplified comparison)
    arm = np.random.choice(range(6))
    engagement = get_engagement(initial_arms[arm])
    random_engagement.append(engagement)

bandit_total = df['engagement'].sum()
random_total = sum(random_engagement)

print("\n" + "="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(f"\nCreator Bandit Playbook:")
print(f"  Total engagement: {bandit_total:.2f}")
print(f"  Avg per post: {bandit_total/len(history):.1%}")

print(f"\nRandom Publishing:")
print(f"  Total engagement: {random_total:.2f}")
print(f"  Avg per post: {random_total/len(random_engagement):.1%}")

improvement = 100 * (bandit_total - random_total) / random_total
print(f"\n🎯 Bandit improvement: +{improvement:.1f}% more engagement")
print(f"   ({bandit_total - random_total:.2f} additional engagement over 52 weeks)")

## Modify This

Experiment with the playbook:

1. **Longer exploration:** Try `exploration_weeks = 6` - does it improve learning?
2. **Higher exploration rate:** Change `exploration_rate = 0.30` (30% explore)
3. **More posts:** Increase `posts_per_week = 7` (daily publishing)
4. **Retire more aggressively:** Drop 3 worst arms at week 26
5. **Multiple retirements:** Add another retirement phase at week 39

In [ ]:
# YOUR EXPERIMENTS HERE
# Copy the code above and modify the playbook parameters

## What's Next?

You just ran the complete Creator Bandit Playbook!

**Continue learning:**
- `00_your_first_bandit.ipynb` - Thompson Sampling basics
- `01_ab_test_vs_bandit.ipynb` - Why bandits beat A/B tests
- `04_algorithm_comparison.ipynb` - Compare different bandit algorithms

**Deep dive:**
- **Module 2:** Thompson Sampling theory and Beta-Bernoulli conjugacy
- **Module 6:** Non-stationary bandits (content trends change!)
- **Module 7:** Exploration strategies and budgets
- **Recipe:** `exploration_strategies.py` - Different ways to balance explore/exploit

**Key insights from the playbook:**
1. Start with exploration to gather data
2. Tilt toward winners but maintain exploration budget
3. Regularly retire underperformers and test new ideas
4. Track engagement to validate your strategy is working